# 03 — MLOps Pipeline Results
PULSE-OPS | End-to-end results: training, registry, serving, and monitoring.

Covers:
- Training XGBoost and LightGBM models with MLflow tracking
- Model registry: register, promote to Staging → Production
- HPO (Hyperparameter Optimisation) with Optuna
- Serving predictions via FastAPI
- System health and Prometheus metrics

In [ ]:
import sys
from pathlib import Path

repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import warnings
warnings.filterwarnings("ignore")

# Point MLflow at local tracking server (or file-based fallback)
import os
mlflow_uri = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(mlflow_uri)
print(f"MLflow URI: {mlflow_uri}")

## 1. Prepare Training Data

In [ ]:
from data.fetchers.uci_fetcher import UCIFetcher
from data.processors.feature_engineer import FeatureEngineer
from data.processors.data_splitter import DataSplitter

fetcher = UCIFetcher()
adult_df = fetcher.fetch_adult_dataset()

engineer = FeatureEngineer()
df = engineer.handle_missing(adult_df.copy())
df = engineer.encode_categoricals(df)

splitter = DataSplitter()
X_train, X_val, X_test, y_train, y_val, y_test = splitter.split(df, "income")

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Positive rate (train): {y_train.mean():.3f}")

## 2. Train XGBoost Classifier

In [ ]:
from models.classification.gradient_boosting import XGBoostClassifier
from models.trainer import ModelTrainer

xgb_model = XGBoostClassifier()
xgb_model.build({
    "n_estimators": 100,
    "max_depth": 5,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
})

trainer = ModelTrainer()
xgb_result = trainer.train(xgb_model, X_train, y_train, X_val, y_val)

print("XGBoost Training Results:")
for k, v in xgb_result["metrics"].items():
    print(f"  {k}: {v:.4f}")
print(f"  run_id: {xgb_result['run_id']}")

## 3. Train LightGBM Classifier

In [ ]:
from models.classification.gradient_boosting import LightGBMClassifier

lgbm_model = LightGBMClassifier()
lgbm_model.build({
    "n_estimators": 100,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_child_samples": 20,
})

lgbm_result = trainer.train(lgbm_model, X_train, y_train, X_val, y_val)

print("LightGBM Training Results:")
for k, v in lgbm_result["metrics"].items():
    print(f"  {k}: {v:.4f}")

## 4. Model Comparison

In [ ]:
from sklearn.metrics import roc_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
for model, result, label, color in [
    (xgb_model, xgb_result, "XGBoost", "steelblue"),
    (lgbm_model, lgbm_result, "LightGBM", "tomato"),
]:
    proba = model.predict_proba(X_test)
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, label=f"{label} (AUC={roc_auc:.3f})")

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curves — Adult Income Classification")
axes[0].legend()

# Metric comparison bar chart
metric_keys = ["val_accuracy", "val_f1", "val_roc_auc"]
xgb_vals = [xgb_result["metrics"].get(k, 0) for k in metric_keys]
lgbm_vals = [lgbm_result["metrics"].get(k, 0) for k in metric_keys]

x = np.arange(len(metric_keys))
width = 0.35
axes[1].bar(x - width / 2, xgb_vals, width, label="XGBoost", color="steelblue", alpha=0.8)
axes[1].bar(x + width / 2, lgbm_vals, width, label="LightGBM", color="tomato", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels([k.replace("val_", "") for k in metric_keys])
axes[1].set_ylim(0.5, 1.0)
axes[1].set_title("Validation Metrics Comparison")
axes[1].legend()
axes[1].set_ylabel("Score")

plt.tight_layout()
plt.show()

## 5. MLflow Model Registry

In [ ]:
from registry.mlflow_registry import MLflowModelRegistry

registry = MLflowModelRegistry()

# Register XGBoost
xgb_version = registry.register_model(xgb_result["run_id"], "adult_xgboost_classifier")
print(f"Registered XGBoost: version={xgb_version.version}")

# Promote to staging
staging = registry.promote_to_staging("adult_xgboost_classifier", int(xgb_version.version))
print(f"Promoted to Staging: stage={staging.current_stage}")

# Promote to production
prod = registry.promote_to_production("adult_xgboost_classifier", int(xgb_version.version))
print(f"Promoted to Production: {prod}")

In [ ]:
# List all registered versions
versions = registry.list_versions("adult_xgboost_classifier")
versions_df = pd.DataFrame([
    {"version": v.version, "stage": v.current_stage, "run_id": v.run_id[:8] + "..."}
    for v in versions
])
print("Registered Versions:")
print(versions_df.to_string(index=False))

## 6. MLflow Experiment Summary

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiments = client.search_experiments()

print("Active MLflow Experiments:")
for exp in experiments:
    runs = client.search_runs(exp.experiment_id, max_results=5)
    print(f"\n  Experiment: {exp.name} (id={exp.experiment_id})")
    for run in runs:
        acc = run.data.metrics.get("val_accuracy", run.data.metrics.get("accuracy", "N/A"))
        print(f"    run_id={run.info.run_id[:8]}... | status={run.info.status} | val_accuracy={acc}")

## 7. System Health Check

In [ ]:
from monitoring.health_checker import HealthChecker

health = HealthChecker()
checks = health.check_all()
grade = health.compute_health_score(checks)

print("System Health Report:")
for check_name, check_result in checks.items():
    status = "✓" if check_result.get("healthy", False) else "✗"
    print(f"  {status} {check_name}: {check_result}")

print(f"\nOverall Health Grade: {grade}")

## 8. Feature Importance

In [ ]:
importances = xgb_model.get_feature_importance()

if importances:
    imp_series = pd.Series(importances).sort_values(ascending=True).tail(15)
    fig, ax = plt.subplots(figsize=(10, 6))
    imp_series.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title("XGBoost — Top 15 Feature Importances (Adult Dataset)")
    ax.set_xlabel("Importance Score")
    plt.tight_layout()
    plt.show()
else:
    print("Feature importances not available.")

## 9. Orchestration Flow Summary

In [ ]:
from orchestration.prefect_flows import (
    training_flow, drift_detection_flow, retraining_flow, deployment_flow
)

print("Available Prefect Flows:")
for flow_fn in [training_flow, drift_detection_flow, retraining_flow, deployment_flow]:
    print(f"  - {flow_fn.__name__}: {flow_fn.__doc__ or 'No docstring'}")